# Sousse Mobility Study — Notebook 01
**First model: Decision Tree classifier on survey data**

- Dataset: 52 survey responses, KK → Beb Bhar corridor
- Target: predict whether a passenger experienced a full louage (Case 1)
- Result: 45.45% accuracy vs 63.64% baseline — insufficient data conclusion
- Next step: collect 150+ responses and retrain

Author: Mariem Belaid | Sousse, Tunisia

In [3]:
import pandas as pd
df = pd.read_csv('survey_file_Mobility_sousse.csv')
df.head()

,Timestamp,station,distination,frequency,time_slot,wait_time,case,ما هو أسوأ يوم في الأسبوع من ناحية الانتظار؟,هل تستخدم أحياناً الحافلة بدل التاكسي الجماعي؟,لماذا تفضّل التاكسي الجماعي على الحافلة؟,ما الوقت أو المكان الأصعب في رأيك ؟,ما الحل الذي تقترحه ؟
0,06/06/2026 18:18:26,قمعون أكودة,المنشية حمام سوسة,كل يوم,الصباح 07:00–09:00,15-30 min,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",الأحد,نادراً,الحافلة لا تأتي في وقتها,Akouda,Nchri krhba🙂
1,06/06/2026 18:27:02,مفترق بانوراما,مفترق موبلاتكس حمام سوسة,مرة أو مرتين,الضحى 09:00–11:00,5-15 min,يمر تاكسي جماعي ممتلئ,الأحد,نعم أحياناً,الحافلة لا تأتي في وقتها,بانوراما,NaN
2,06/06/2026 18:30:13,قمعون أكودة,باب بحر,مرة أو مرتين,الضحى 09:00–11:00,15-30 min,يمر تاكسي جماعي ممتلئ,الأحد,نادراً,التاكسي الجماعي أسرع,في العشية وقت مرواح الخدامة,إنشاء محطة بأكودة
3,06/06/2026 18:36:21,محطة مستشفى فرحات حشاد,محطة مستشفى فرحات حشاد,3 الى 5 مرات,بعد الظهر 16:00–18:00,more than 30 min,"يمر تاكسي جماعي ممتلئ, لا يمر تاكسي جماعي لفتر...",الاثنين,نعم أحياناً,الحافلة لا تأتي في وقتها,باب بحر على الساعة الخامسة مساء,تكثيف دوريات شرطة المرور ، تسهيل الإجراءات لتق...
4,06/06/2026 18:40:24,معهد علي بورقيبة القلعة الكبرى,خزامة,كل يوم,الصباح 07:00–09:00,more than 30 min,يمر تاكسي جماعي لكن الناس يتدافعون ولا تجد مكانا,السبت,لا أبداً,NaN,رومبوان الرزڨي,محطة تاكسي خاصة


In [4]:
print(df.shape)
print(df.columns.tolist())

(52, 12)
['Timestamp', 'station', 'distination', 'frequency', 'time_slot', 'wait_time', 'case    ', '  ما هو أسوأ يوم في الأسبوع من ناحية الانتظار؟  ', '  هل تستخدم أحياناً الحافلة بدل التاكسي الجماعي؟  ', '    لماذا تفضّل التاكسي الجماعي على الحافلة؟ ', 'ما الوقت أو المكان الأصعب في رأيك ؟ ', 'ما الحل الذي تقترحه ؟ ']


In [5]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Timestamp', 'station', 'distination', 'frequency', 'time_slot', 'wait_time', 'case', 'ما هو أسوأ يوم في الأسبوع من ناحية الانتظار؟', 'هل تستخدم أحياناً الحافلة بدل التاكسي الجماعي؟', 'لماذا تفضّل التاكسي الجماعي على الحافلة؟', 'ما الوقت أو المكان الأصعب في رأيك ؟', 'ما الحل الذي تقترحه ؟']


In [6]:
df = df.rename(columns={
    'ما هو أسوأ يوم في الأسبوع من ناحية الانتظار؟': 'worst_day',
    'هل تستخدم أحياناً الحافلة بدل التاكسي الجماعي؟': 'uses_bus',
    'لماذا تفضّل التاكسي الجماعي على الحافلة؟': 'why_prefer_louage',
    'ما الوقت أو المكان الأصعب في رأيك ؟': 'hardest_time_place',
    'ما الحل الذي تقترحه ؟': 'suggested_solution',
})

print(df.columns.tolist())

['Timestamp', 'station', 'distination', 'frequency', 'time_slot', 'wait_time', 'case', 'worst_day', 'uses_bus', 'why_prefer_louage', 'hardest_time_place', 'suggested_solution']


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Timestamp           52 non-null     object
 1   station             52 non-null     object
 2   distination         52 non-null     object
 3   frequency           52 non-null     object
 4   time_slot           52 non-null     object
 5   wait_time           52 non-null     object
 6   case                52 non-null     object
 7   worst_day           52 non-null     object
 8   uses_bus            52 non-null     object
 9   why_prefer_louage   39 non-null     object
 10  hardest_time_place  40 non-null     object
 11  suggested_solution  37 non-null     object
dtypes: object(12)
memory usage: 5.0+ KB


In [8]:
for col in ['station', 'distination', 'frequency', 'time_slot', 'wait_time', 'case', 'worst_day', 'uses_bus']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- station ---
station
(la poste) البريد  القلعة الكبرى          11
قمعون أكودة                                8
معهد علي بورقيبة القلعة الكبرى             7
سير عويتي القلعة الكبرى                    7
المراح القلعة الكبرى                       5
الوردة أكودة                               5
الحي الجديد (مخبزة زايد) القلعة الكبرى     4
باب بحر                                    2
مفترق بانوراما                             1
محطة مستشفى فرحات حشاد                     1
المنشية حمام سوسة                          1
Name: count, dtype: int64

--- distination ---
distination
باب بحر                                   18
المنشية حمام سوسة                         16
معهد علي بورقيبة القلعة الكبرى             4
خزامة                                      3
مفترق بانوراما                             2
محطة مستشفى فرحات حشاد                     2
قمعون أكودة                                2
مفترق موبلاتكس حمام سوسة                   1
مفترق حنبعل أكودة                          1
(la poste) البري

In [9]:
df['case_full'] = df['case'].str.contains('ممتلئ', na=False).astype(int)
df['case_wrongline'] = df['case'].str.contains('سهلول', na=False).astype(int)
df['case_nogap'] = df['case'].str.contains('لا يمر', na=False).astype(int)
df['case_rush'] = df['case'].str.contains('يتدافعون', na=False).astype(int)

df[['case', 'case_full', 'case_wrongline', 'case_nogap', 'case_rush']].head(10)

,case,case_full,case_wrongline,case_nogap,case_rush
0,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1
1,يمر تاكسي جماعي ممتلئ,1,0,0,0
2,يمر تاكسي جماعي ممتلئ,1,0,0,0
3,"يمر تاكسي جماعي ممتلئ, لا يمر تاكسي جماعي لفتر...",1,0,1,1
4,يمر تاكسي جماعي لكن الناس يتدافعون ولا تجد مكانا,0,0,0,1
5,يمر تاكسي جماعي لكن الناس يتدافعون ولا تجد مكانا,0,0,0,1
6,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1
7,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي اتجاهه ...",1,1,0,0
8,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي اتجاهه ...",1,1,1,1
9,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1


In [10]:
df[['case', 'case_full', 'case_wrongline', 'case_nogap', 'case_rush']].head(10)

,case,case_full,case_wrongline,case_nogap,case_rush
0,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1
1,يمر تاكسي جماعي ممتلئ,1,0,0,0
2,يمر تاكسي جماعي ممتلئ,1,0,0,0
3,"يمر تاكسي جماعي ممتلئ, لا يمر تاكسي جماعي لفتر...",1,0,1,1
4,يمر تاكسي جماعي لكن الناس يتدافعون ولا تجد مكانا,0,0,0,1
5,يمر تاكسي جماعي لكن الناس يتدافعون ولا تجد مكانا,0,0,0,1
6,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1
7,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي اتجاهه ...",1,1,0,0
8,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي اتجاهه ...",1,1,1,1
9,"يمر تاكسي جماعي ممتلئ, يمر تاكسي جماعي لكن الن...",1,0,0,1


In [11]:
from sklearn.preprocessing import LabelEncoder
model_df = df[['station', 'time_slot', 'frequency', 'wait_time']].copy()
print(model_df.isnull().sum())


station      0
time_slot    0
frequency    0
wait_time    0
dtype: int64


In [12]:
encoders = {}
for col in ['station', 'time_slot', 'frequency', 'wait_time']:
    le = LabelEncoder()
    model_df[col + '_enc'] = le.fit_transform(model_df[col])
    encoders[col] = le

model_df.head(10)

,station,time_slot,frequency,wait_time,station_enc,time_slot_enc,frequency_enc,wait_time_enc
0,قمعون أكودة,الصباح 07:00–09:00,كل يوم,15-30 min,7,0,1,0
1,مفترق بانوراما,الضحى 09:00–11:00,مرة أو مرتين,5-15 min,10,2,2,1
2,قمعون أكودة,الضحى 09:00–11:00,مرة أو مرتين,15-30 min,7,2,2,0
3,محطة مستشفى فرحات حشاد,بعد الظهر 16:00–18:00,3 الى 5 مرات,more than 30 min,8,5,0,3
4,معهد علي بورقيبة القلعة الكبرى,الصباح 07:00–09:00,كل يوم,more than 30 min,9,0,1,3
5,المراح القلعة الكبرى,الصباح 07:00–09:00,3 الى 5 مرات,more than 30 min,2,0,0,3
6,المراح القلعة الكبرى,بعد الظهر 16:00–18:00,3 الى 5 مرات,5-15 min,2,5,0,1
7,(la poste) البريد القلعة الكبرى,الصباح 07:00–09:00,كل يوم,less than 5 min,0,0,1,2
8,(la poste) البريد القلعة الكبرى,الصباح 07:00–09:00,كل يوم,more than 30 min,0,0,1,3
9,(la poste) البريد القلعة الكبرى,الضحى 09:00–11:00,كل يوم,5-15 min,0,2,1,1


In [13]:
from sklearn.model_selection import train_test_split
X = model_df[['station_enc','time_slot_enc', 'frequency_enc']]
y = model_df['wait_time_enc']
X_train , X_test , y_train , y_test = train_test_split(X , y, test_size=0.2, random_state=42 )
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (41, 3)
Test set size: (11, 3)


In [17]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
for depth in [1, 2, 3, 4, 5, 6, 7, 8]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"depth={depth}  train_acc={train_acc:.2%}  test_acc={test_acc:.2%}")

depth=1  train_acc=56.10%  test_acc=36.36%
depth=2  train_acc=56.10%  test_acc=36.36%
depth=3  train_acc=63.41%  test_acc=45.45%
depth=4  train_acc=65.85%  test_acc=36.36%
depth=5  train_acc=70.73%  test_acc=36.36%
depth=6  train_acc=75.61%  test_acc=27.27%
depth=7  train_acc=80.49%  test_acc=27.27%
depth=8  train_acc=82.93%  test_acc=27.27%


In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test , y_pred)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 45.45%


In [19]:
from sklearn.metrics import confusion_matrix, classification_report

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

labels = encoders['wait_time'].classes_

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)
print("\nLabels order:", labels)

print("\n", classification_report(y_test, y_pred, target_names=labels))

Confusion matrix:
[[0 1 0 2]
 [0 0 0 1]
 [1 1 0 0]
 [0 0 0 5]]

Labels order: ['15-30 min' '5-15 min' 'less than 5 min' 'more than 30 min']

                   precision    recall  f1-score   support

       15-30 min       0.00      0.00      0.00         3
        5-15 min       0.00      0.00      0.00         1
 less than 5 min       0.00      0.00      0.00         2
more than 30 min       0.62      1.00      0.77         5

        accuracy                           0.45        11
       macro avg       0.16      0.25      0.19        11
    weighted avg       0.28      0.45      0.35        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
print(df['case_full'].value_counts())
print(df['case_full'].value_counts(normalize=True))

case_full
1    35
0    17
Name: count, dtype: int64
case_full
1    0.673077
0    0.326923
Name: proportion, dtype: float64


In [21]:

y_binary = model_df_full = df['case_full']
X = model_df[['station_enc', 'time_slot_enc', 'frequency_enc']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print("Train distribution:\n", y_train.value_counts(normalize=True))
print("Test distribution:\n", y_test.value_counts(normalize=True))

Train distribution:
 case_full
1    0.682927
0    0.317073
Name: proportion, dtype: float64
Test distribution:
 case_full
1    0.636364
0    0.363636
Name: proportion, dtype: float64


In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")
print(f"Baseline (always predict 1): {y_test.value_counts(normalize=True)[1]:.2%}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n", classification_report(y_test, y_pred, target_names=['Not Full (0)', 'Full (1)']))

Accuracy: 45.45%
Baseline (always predict 1): 63.64%

Confusion matrix:
[[1 3]
 [3 4]]

               precision    recall  f1-score   support

Not Full (0)       0.25      0.25      0.25         4
    Full (1)       0.57      0.57      0.57         7

    accuracy                           0.45        11
   macro avg       0.41      0.41      0.41        11
weighted avg       0.45      0.45      0.45        11



In [23]:
zone_map = {
    'دردور القلعة الكبرى': 'Kalaa_Kebira',
    'سير عويتي القلعة الكبرى': 'Kalaa_Kebira',
    'المراح القلعة الكبرى': 'Kalaa_Kebira',
    '(la poste) البريد  القلعة الكبرى': 'Kalaa_Kebira',
    'الحي الجديد (مخبزة زايد) القلعة الكبرى': 'Kalaa_Kebira',
    'معهد علي بورقيبة القلعة الكبرى': 'Kalaa_Kebira',
    'الوردة أكودة': 'Akouda',
    'قمعون أكودة': 'Akouda',
    'مفترق حنبعل أكودة': 'Akouda',
    'مفترق موبلاتكس حمام سوسة': 'Hammam_Sousse',
    'سيدي سالم حمام سوسة': 'Hammam_Sousse',
    'المنشية حمام سوسة': 'Hammam_Sousse',
    'خزامة': 'Sousse_City',
    'مفترق بانوراما': 'Sousse_City',
    'محطة مستشفى فرحات حشاد': 'Sousse_City',
    'باب بحر': 'Sousse_City',
}

df['zone'] = df['station'].map(zone_map)

print(df['zone'].value_counts())
print("\nUnmapped (check for typos):", df[df['zone'].isnull()]['station'].unique())

zone
Kalaa_Kebira     34
Akouda           13
Sousse_City       4
Hammam_Sousse     1
Name: count, dtype: int64

Unmapped (check for typos): []


In [24]:
df['zone'] = df['zone'].replace('Hammam_Sousse', 'Sousse_City')

print(df['zone'].value_counts())

zone
Kalaa_Kebira    34
Akouda          13
Sousse_City      5
Name: count, dtype: int64


In [26]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
le_zone = LabelEncoder()
df['zone_enc'] = le_zone.fit_transform(df['zone'])

print("Zone encoding:")
for zone, code in zip(le_zone.classes_, range(len(le_zone.classes_))):
    print(f"  {code} = {zone}")


le_time = LabelEncoder()
le_freq = LabelEncoder()

df['time_slot_enc'] = le_time.fit_transform(df['time_slot'])
df['frequency_enc'] = le_freq.fit_transform(df['frequency'])


print(df[['zone_enc', 'time_slot_enc', 'frequency_enc']].head())
X = df[['zone_enc', 'time_slot_enc', 'frequency_enc']]
y = df['case_full']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
baseline = y_test.value_counts(normalize=True)[1]
print(f"\nAccuracy:  {accuracy:.2%}")
print(f"Baseline:  {baseline:.2%}")
print(f"Beat baseline by: {(accuracy - baseline):.2%}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n", classification_report(y_test, y_pred,
      target_names=['Not Full (0)', 'Full (1)']))

Zone encoding:
  0 = Akouda
  1 = Kalaa_Kebira
  2 = Sousse_City
   zone_enc  time_slot_enc  frequency_enc
0         0              0              1
1         2              2              2
2         0              2              2
3         2              5              0
4         1              0              1

Accuracy:  45.45%
Baseline:  63.64%
Beat baseline by: -18.18%

Confusion matrix:
[[1 3]
 [3 4]]

               precision    recall  f1-score   support

Not Full (0)       0.25      0.25      0.25         4
    Full (1)       0.57      0.57      0.57         7

    accuracy                           0.45        11
   macro avg       0.41      0.41      0.41        11
weighted avg       0.45      0.45      0.45        11

